In [10]:
import os
import tarfile
import random
import re
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import download_asset
from torch.utils.data import Dataset, DataLoader
from jiwer import wer
from tqdm.auto import tqdm
import whisper

In [11]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [12]:
class Config:
    sr = 16000
    batch_size = 4
    epochs = 50
    lr = 1e-3
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tar_path = "ru_train_0_19.tar"
    tsv_path = "train(1).tsv"
    extract_dir = "./ru_train_data"
    n_fft = 1024
    hop_length = 256
    target_samples = 192 * 256 + 1024

In [13]:
def init_unitary_complex(shape):
    fan_in = shape[1] * shape[2] * shape[3]
    fan_out = shape[0] * shape[2] * shape[3]
    scale = math.sqrt(2.0 / (fan_in + fan_out))
    
    re_w = np.random.normal(size=(shape[0], fan_in))
    im_w = np.random.normal(size=(shape[0], fan_in))
    x = re_w + 1j * im_w
    
    if shape[0] < fan_in:
        q, _ = np.linalg.qr(x.T)
        q = q.T
    else:
        q, _ = np.linalg.qr(x)
        
    q *= scale
    return torch.from_numpy(q.real).float(), torch.from_numpy(q.imag).float()

class ComplexConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.weight_r = nn.Parameter(torch.empty(out_channels, in_channels, *kernel_size))
        self.weight_i = nn.Parameter(torch.empty(out_channels, in_channels, *kernel_size))
        
        re_init, im_init = init_unitary_complex(self.weight_r.shape)
        self.weight_r.data.copy_(re_init.view_as(self.weight_r))
        self.weight_i.data.copy_(im_init.view_as(self.weight_i))

    def forward(self, x):
        x_r, x_i = x.real, x.imag
        out_r = F.conv2d(x_r, self.weight_r, stride=self.stride, padding=self.padding) - \
                F.conv2d(x_i, self.weight_i, stride=self.stride, padding=self.padding)
        out_i = F.conv2d(x_r, self.weight_i, stride=self.stride, padding=self.padding) + \
                F.conv2d(x_i, self.weight_r, stride=self.stride, padding=self.padding)
        return torch.complex(out_r, out_i)

class ComplexConvTranspose2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.stride = stride if isinstance(stride, tuple) else (stride, stride)
        self.padding = padding if isinstance(padding, tuple) else (padding, padding)
        self.kernel_size = kernel_size
        
        self.weight_r = nn.Parameter(torch.empty(in_channels, out_channels, *kernel_size))
        self.weight_i = nn.Parameter(torch.empty(in_channels, out_channels, *kernel_size))

        re_init, im_init = init_unitary_complex((in_channels, out_channels, *kernel_size))
        self.weight_r.data.copy_(re_init.view_as(self.weight_r))
        self.weight_i.data.copy_(im_init.view_as(self.weight_i))

    def forward(self, x, output_size=None):
        x_r, x_i = x.real, x.imag
        
        out_padding = (0, 0)
        if output_size is not None:
            h_out_base = (x_r.shape[2] - 1) * self.stride[0] - 2 * self.padding[0] + self.kernel_size[0]
            w_out_base = (x_r.shape[3] - 1) * self.stride[1] - 2 * self.padding[1] + self.kernel_size[1]
            
            out_padding = (
                max(0, output_size[2] - h_out_base),
                max(0, output_size[3] - w_out_base)
            )

        out_r = F.conv_transpose2d(x_r, self.weight_r, stride=self.stride, padding=self.padding, output_padding=out_padding) - \
                F.conv_transpose2d(x_i, self.weight_i, stride=self.stride, padding=self.padding, output_padding=out_padding)
        out_i = F.conv_transpose2d(x_r, self.weight_i, stride=self.stride, padding=self.padding, output_padding=out_padding) + \
                F.conv_transpose2d(x_i, self.weight_r, stride=self.stride, padding=self.padding, output_padding=out_padding)

        return torch.complex(out_r, out_i)

class ComplexBatchNorm2d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.gamma_rr = nn.Parameter(torch.ones(num_features, 1, 1))
        self.gamma_ii = nn.Parameter(torch.ones(num_features, 1, 1))
        self.gamma_ri = nn.Parameter(torch.zeros(num_features, 1, 1))
        self.beta_r = nn.Parameter(torch.zeros(num_features, 1, 1))
        self.beta_i = nn.Parameter(torch.zeros(num_features, 1, 1))
        self.register_buffer('run_mu_r', torch.zeros(num_features))
        self.register_buffer('run_mu_i', torch.zeros(num_features))
        self.register_buffer('run_Vrr', torch.ones(num_features))
        self.register_buffer('run_Vii', torch.ones(num_features))
        self.register_buffer('run_Vri', torch.zeros(num_features))

    def forward(self, x):
        x_r, x_i = x.real, x.imag
        if self.training:
            mu_r = x_r.mean(dim=(0, 2, 3), keepdim=True)
            mu_i = x_i.mean(dim=(0, 2, 3), keepdim=True)
            x_r_c, x_i_c = x_r - mu_r, x_i - mu_i
            Vrr = (x_r_c ** 2).mean(dim=(0, 2, 3))
            Vii = (x_i_c ** 2).mean(dim=(0, 2, 3))
            Vri = (x_r_c * x_i_c).mean(dim=(0, 2, 3))
            with torch.no_grad():
                self.run_mu_r = (1 - self.momentum) * self.run_mu_r + self.momentum * mu_r.squeeze()
                self.run_mu_i = (1 - self.momentum) * self.run_mu_i + self.momentum * mu_i.squeeze()
                self.run_Vrr = (1 - self.momentum) * self.run_Vrr + self.momentum * Vrr
                self.run_Vii = (1 - self.momentum) * self.run_Vii + self.momentum * Vii
                self.run_Vri = (1 - self.momentum) * self.run_Vri + self.momentum * Vri
        else:
            mu_r, mu_i = self.run_mu_r.view(1, -1, 1, 1), self.run_mu_i.view(1, -1, 1, 1)
            Vrr, Vii, Vri = self.run_Vrr, self.run_Vii, self.run_Vri
            x_r_c, x_i_c = x_r - mu_r, x_i - mu_i

        det = Vrr * Vii - Vri ** 2
        s = torch.sqrt(det + self.eps)
        t = torch.sqrt(Vrr + Vii + 2 * s + self.eps)
        Wrr, Wii, Wri = (Vii + s) / (t * s), (Vrr + s) / (t * s), -Vri / (t * s)
        Wrr, Wii, Wri = Wrr.view(1, -1, 1, 1), Wii.view(1, -1, 1, 1), Wri.view(1, -1, 1, 1)
        
        x_r_w = Wrr * x_r_c + Wri * x_i_c
        x_i_w = Wri * x_r_c + Wii * x_i_c
        out_r = self.gamma_rr * x_r_w + self.gamma_ri * x_i_w + self.beta_r
        out_i = self.gamma_ri * x_r_w + self.gamma_ii * x_i_w + self.beta_i
        return torch.complex(out_r, out_i)

class ComplexLeakyReLU(nn.Module):
    def forward(self, x):
        return torch.complex(F.leaky_relu(x.real, 0.2), F.leaky_relu(x.imag, 0.2))

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride):
        super().__init__()
        self.conv = ComplexConv2d(in_ch, out_ch, kernel_size, stride, padding=(kernel_size[0]//2, kernel_size[1]//2))
        self.bn = ComplexBatchNorm2d(out_ch)
        self.act = ComplexLeakyReLU()

    def forward(self, x): return self.act(self.bn(self.conv(x)))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride, is_last=False):
        super().__init__()
        self.trans_conv = ComplexConvTranspose2d(in_ch, out_ch, kernel_size, stride, padding=(kernel_size[0]//2, kernel_size[1]//2))
        self.is_last = is_last
        if not is_last:
            self.bn = ComplexBatchNorm2d(out_ch)
            self.act = ComplexLeakyReLU()

    def forward(self, x, skip_target):
        x = self.trans_conv(x, output_size=skip_target.size())
        return x if self.is_last else self.act(self.bn(x))

class LargeDCUnet20(nn.Module):
    def __init__(self):
        super().__init__()
        channels = [45] + [90] * 9  
        kernels = [(7, 5), (7, 5)] + [(5, 3)] * 8
        strides = [(2, 2)] * 5 + [(2, 1)] * 4 + [(2, 1)]
        
        self.encoders = nn.ModuleList()
        in_c = 1
        for out_c, k, s in zip(channels, kernels, strides):
            self.encoders.append(EncoderBlock(in_c, out_c, k, s))
            in_c = out_c
            
        self.decoders = nn.ModuleList()
        dec_channels, dec_kernels, dec_strides = channels[::-1], kernels[::-1], strides[::-1]
        for i in range(10):
            in_c = channels[-1] if i == 0 else dec_channels[i] * 2 
            out_c = dec_channels[i + 1] if i < 9 else 1
            self.decoders.append(DecoderBlock(in_c, out_c, dec_kernels[i], dec_strides[i], is_last=(i == 9)))

    def forward(self, x):
        skips = []
        out = x
        for enc in self.encoders:
            out = enc(out)
            skips.append(out)

        rev_skips = skips[::-1]
        targets = skips[:-1][::-1] + [x] 
        
        out = rev_skips[0]

        for i, dec in enumerate(self.decoders):
            if i == 0:
                out = dec(out, targets[i])
            else:
                out = torch.cat([out, rev_skips[i]], dim=1)
                out = dec(out, targets[i])
                
        return out

class SpeechEnhancementPipeline(nn.Module):
    def __init__(self, n_fft=1024, hop_length=256):
        super().__init__()
        self.n_fft, self.hop_length = n_fft, hop_length
        self.dcunet = LargeDCUnet20()
        
        window = torch.hann_window(n_fft)
        n = torch.arange(n_fft).unsqueeze(1)
        k = torch.arange(n_fft // 2 + 1).unsqueeze(0)
        
        W_stft = torch.exp(-2j * math.pi * k * n / n_fft)
        self.register_buffer('stft_weights_r', (W_stft.real * window.unsqueeze(1)).T.unsqueeze(1))
        self.register_buffer('stft_weights_i', (W_stft.imag * window.unsqueeze(1)).T.unsqueeze(1))
        
        scale = torch.ones(n_fft // 2 + 1)
        scale[1:-1] = 2.0
        scale /= n_fft
        
        W_istft_r = (torch.cos(2 * math.pi * k * n / n_fft) * window.unsqueeze(1) * scale.unsqueeze(0) / 1.5).T.unsqueeze(1)
        W_istft_i = (-torch.sin(2 * math.pi * k * n / n_fft) * window.unsqueeze(1) * scale.unsqueeze(0) / 1.5).T.unsqueeze(1)
        self.register_buffer('istft_weights_r', W_istft_r)
        self.register_buffer('istft_weights_i', W_istft_i)

    def forward(self, x):
        original_T = x.size(-1)
        if x.dim() == 2: x = x.unsqueeze(1)
        
        # STFT
        X_r = F.conv1d(x, self.stft_weights_r, stride=self.hop_length)
        X_i = F.conv1d(x, self.stft_weights_i, stride=self.hop_length)
        X_stft = torch.complex(X_r, X_i)
        
        X_cut = X_stft[:, :-1, :].unsqueeze(1) 
        T_f = X_cut.size(-1)
        pad_t = (512 - (T_f % 512)) % 512
        if pad_t > 0: X_cut = F.pad(X_cut, (0, pad_t))
            
        O_cut = self.dcunet(X_cut)
        
        if pad_t > 0: O_cut = O_cut[..., :-pad_t]
        O = F.pad(O_cut, (0, 0, 0, 1)).squeeze(1)
        
        mag_O = torch.abs(O)
        M_hat = torch.tanh(mag_O) * (O / (mag_O + 1e-8))
        Y_hat_stft = M_hat * X_stft
        
        out_r = F.conv_transpose1d(Y_hat_stft.real, self.istft_weights_r, stride=self.hop_length)
        out_i = F.conv_transpose1d(Y_hat_stft.imag, self.istft_weights_i, stride=self.hop_length)
        y_hat = (out_r + out_i).squeeze(1)
        
        if y_hat.size(-1) > original_T: y_hat = y_hat[..., :original_T]
        elif y_hat.size(-1) < original_T: y_hat = F.pad(y_hat, (0, original_T - y_hat.size(-1)))
            
        return y_hat

def wSDRLoss(x, y, y_hat, eps=1e-8):
    def sdr_loss(y_true, y_pred):
        return -torch.sum(y_true * y_pred, dim=-1) / (torch.norm(y_true, dim=-1) * torch.norm(y_pred, dim=-1) + eps)
    
    z, z_hat = x - y, x - y_hat
    y_norm2, z_norm2 = torch.sum(y**2, dim=-1), torch.sum(z**2, dim=-1)
    alpha = y_norm2 / (y_norm2 + z_norm2 + eps)
    
    return (alpha * sdr_loss(y, y_hat) + (1 - alpha) * sdr_loss(z, z_hat)).mean()

In [14]:
def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

def get_snr_scale(signal, noise, snr_db):
    sig_power = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noise_power = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    return torch.sqrt(target_noise_power / (noise_power + 1e-8))

def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)

    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]
    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

In [15]:
if not os.path.exists(Config.extract_dir):
    os.makedirs(Config.extract_dir, exist_ok=True)
    with tarfile.open(Config.tar_path, "r") as tar:
        tar.extractall(path=Config.extract_dir)

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

babble_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
BABBLE_WAVEFORM, sr_b = torchaudio.load(babble_path)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
RIR_WAVEFORM, sr_r = torchaudio.load(rir_path)
RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

/tmp/ipykernel_99573/2387710718.py:9: UserWarning: torchaudio.utils.download.download_asset has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  babble_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packa

In [16]:
class ExactSequenceDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        
        file_list = []
        for r, _, fs in os.walk(data_dir):
            for f in fs:
                if f.endswith('.mp3') and f in ref_dict:
                    file_list.append(os.path.join(r, f))
        self.files = sorted(file_list)
        
        self.target_samples = int(3.0 * Config.sr)

    def __len__(self): 
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        wav, sr = torchaudio.load(file_path)
        
        if sr != Config.sr: 
            wav = T.Resample(sr, Config.sr)(wav)
            
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
            
        if self.is_train:
            if wav.shape[-1] > self.target_samples:
                s = random.randint(0, wav.shape[-1] - self.target_samples)
                wav = wav[:, s:s+self.target_samples]
            else: 
                wav = F.pad(wav, (0, self.target_samples - wav.shape[-1]))
        else:
            if wav.shape[-1] > self.target_samples:
                wav = wav[:, :self.target_samples]
            else:
                wav = F.pad(wav, (0, self.target_samples - wav.shape[-1]))
                
        clean = wav
        noisy = apply_noise(clean) if self.is_train else clean
        
        return noisy.squeeze(0), clean.squeeze(0), self.ref_dict[os.path.basename(file_path)]

def exact_collate_fn(batch):
    noisy, clean, texts = zip(*batch)
    noisy = torch.stack(noisy).squeeze(1)
    clean = torch.stack(clean).squeeze(1)

    pad_len = Config.target_samples - noisy.shape[-1]
    if pad_len > 0:
        noisy = F.pad(noisy, (0, pad_len))
        clean = F.pad(clean, (0, pad_len))

    return noisy, clean, list(texts)

In [18]:
dataset = ExactSequenceDataset(Config.extract_dir, reference_dict, is_train=True)
generator = torch.Generator().manual_seed(42)
train_size = int(0.9 * len(dataset))
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size], generator=generator)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True, collate_fn=exact_collate_fn)

model = SpeechEnhancementPipeline(n_fft=Config.n_fft, hop_length=Config.hop_length).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)

In [39]:
for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for noisy, clean, _ in pbar:
        noisy, clean = noisy.to(Config.device), clean.to(Config.device)
        optimizer.zero_grad()

        y_hat = model(noisy)

        loss = wSDRLoss(noisy, clean, y_hat)

        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"wSDR Loss": f"{loss.item():.4f}"})

    if epoch % 5 == 0:
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
        }, f"dcunet_checkpoint_{epoch}.pth")

Epoch 1:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/5954 [00:00<?, ?it/s]

...

Epoch 46:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 47:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 48:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 49:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 50:   0%|          | 0/5954 [00:00<?, ?it/s]

In [19]:
checkpoint = torch.load("dcunet_checkpoint_50.pth", map_location=torch.device('cuda'))
model.load_state_dict(checkpoint['model_state_dict'])

seed_everything(42)

In [56]:
def evaluate_dcunet(model, device, val_dataset, limit=20):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": []} for n in noise_types}

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="WER Eval"):
            _, clean_wav, ref_text = val_dataset[idx]
            ref_text = clean_text(ref_text)

            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav, force_type=n_type, file_seed=idx)

                pad_len = Config.target_samples - noisy_wav.shape[-1]
                if pad_len > 0:
                    noisy_input = F.pad(noisy_wav, (0, pad_len)).to(device)
                else:
                    noisy_input = noisy_wav.to(device)

                denoised_wav = model(noisy_input)

                if pad_len > 0:
                    denoised_wav = denoised_wav[..., :-pad_len]

                t_n = asr.transcribe(noisy_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']

                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))

    print(f"\n{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denoised':<10}")
    for n in noise_types:
        wn, wd = np.mean(stats[n]["wer_n"]), np.mean(stats[n]["wer_d"])
        print(f"{n:<10} | {wn:<10.4f} | {wd:<10.4f}")

evaluate_dcunet(model=model, device=Config.device, val_dataset=val_ds, limit=20)

WER Eval:   0%|          | 0/20 [00:00<?, ?it/s]


Noise Type | WER Noisy  | WER Denoised
babble     | 0.5376     | 0.4614    
rir        | 1.0000     | 0.9723    
white      | 0.5670     | 0.6627    


In [20]:
from jiwer import process_words

def evaluate_and_listen_components(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    noise_types = ['babble', 'rir', 'white']
    
    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}
    
    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
    
    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            
            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav, force_type=n_type, file_seed=idx)
                
                pad_len = Config.target_samples - noisy_wav.shape[-1]
                if pad_len > 0:
                    noisy_input = F.pad(noisy_wav, (0, pad_len)).to(device)
                else:
                    noisy_input = noisy_wav.to(device)
                
                denoised_wav = model(noisy_input)
                
                if pad_len > 0:
                    denoised_wav = denoised_wav[..., :-pad_len]
                
                noisy_np = noisy_wav.squeeze().cpu().numpy()
                denoised_np = denoised_wav.squeeze().cpu().numpy()
                
                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']
                
                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)
                
                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits
                
                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    stats[n_type]["wer_n"].append(0.0)
                    stats[n_type]["s_n"].append(0.0)
                    stats[n_type]["d_n"].append(0.0)
                    stats[n_type]["i_n"].append(0.0)
                
                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits
                
                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    stats[n_type]["wer_d"].append(0.0)
                    stats[n_type]["s_d"].append(0.0)
                    stats[n_type]["d_d"].append(0.0)
                    stats[n_type]["i_d"].append(0.0)
    
    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print(header)
    print("-" * 65)
    
    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])
        
        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])
        
        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"
        
        print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")

seed_everything(42)
evaluate_and_listen_components(model=model, device=Config.device, val_dataset=val_ds, limit=20)

Evaluation (Macro-average):   0%|          | 0/20 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Noise    | WER_N   (S/D/I) | WER_D   (S/D/I) | Gain WER
-----------------------------------------------------------------
babble   | 0.5376 (0.1392/0.3875/0.0108) | 0.4614 (0.1070/0.3499/0.0045) | 0.0761  
rir      | 1.0000 (0.3116/0.6884/0.0000) | 0.9723 (0.2762/0.6860/0.0100) | 0.0277  
white    | 0.5670 (0.2060/0.3548/0.0063) | 0.6627 (0.2633/0.3886/0.0108) | -0.0957 
